# Train the land-cover / buildings segmenter (OpenEarthMap)

**Goal:** given ONE satellite/aerial image, label every pixel as one of eight land-cover classes -- **building, road, tree, water, agriculture land, rangeland (grass/scrub), developed space (paved lots, plazas, yards) or bareland** -- so the app can say how much of a scene is roofs, roads, vegetation or water, count the distinct buildings, and name the roof colours from the pixels inside the building mask. The app's existing specialists cannot do any of this: the object detector was trained on object datasets (DOTA / DIOR / VRSBench) that have no building, road or vegetation class at all, and the captioner writes about them but cannot count. A user asked for exactly this ("why no roads, cars, buildings, building colour, building count?") after a street scene came back with only "no airplanes, ships or storage tanks found".

**Data:** [OpenEarthMap](https://open-earth-map.org/) (CC BY-NC-SA 4.0, research use) via the Kaggle copy `aletbm/global-land-cover-mapping-openearthmap`: 2,303 train + 384 val 1024x1024 RGB tiles at 0.25-0.5 m per pixel from 75+ regions on five continents, hand-labelled with 8 classes. It is attached to this notebook as an input, so there is no download step.

**Model:** `segmentation_models_pytorch` U-Net with an ImageNet-pretrained ResNet34 encoder -- the same recipe as the app's water-body model, which trained cleanly on Kaggle. Cross-entropy plus Dice (Dice keeps the rare classes -- bareland, water -- from being ignored).

## What was checked before writing this (so the cells below aren't guesses)

* **Layout** (listed through the Kaggle API): `images/{train,val,test}/<region>_<n>.tif` and `label/{train,val}/<region>_<n>.tif`; there are no test labels. Images are RGB TIFF (1024x1024; a few 1000x1000); labels are 8-bit single-band.
* **Label encoding**, checked by rendering label overlays on eight real tiles and looking at them: `0` = unlabelled (ignored), `1` bareland, `2` rangeland, `3` developed space, `4` road, `5` tree, `6` water, `7` agriculture land, `8` building. On the sample the roofs of a dense Peruvian neighbourhood are exactly class 8, streets class 4, paved lots class 3, an island's sea class 6.
* **Class balance** on that sample: developed space 17.6%, tree 25.6%, building 18.9%, rangeland 15.6%, agriculture 10.1%, road 6.0%, water 5.3%, bareland 0.5% -- hence the Dice term.
* **Ground resolution varies 5-7x between the data and the app's inputs** (0.25-0.5 m per pixel here; a zoomed street view is ~0.2 m, a 1.4 km capture ~1.4 m), so training crops are taken at a random scale (a 256-1000 px window resized to 512, log-uniform) rather than at one fixed scale, and the final evaluation reports the score at native scale *and* at half scale.

## Metrics

Per-class IoU and mIoU on the 384 validation tiles, plus the three numbers the app will actually surface: **area-fraction error per class** (percentage points of the scene), **building-count error** (connected components of the predicted vs true building mask -- touching buildings merge, so this is "distinct footprints", not houses), and the score at half scale. Numbers here say the segmenter works on OpenEarthMap imagery; whether it works on Esri captures is checked separately by eye on real scenes.

Run cells top to bottom. Kaggle: enable **Internet** (for the ImageNet encoder weights) and a **GPU**, and attach the OpenEarthMap dataset as an input.

## 0. GPU compatibility check

**Found live, via an actual failed run on Kaggle**: this session's preinstalled PyTorch build
(2.10.0+cu128) only supports CUDA compute capabilities sm_70 and up -- it silently drops support
for the Pascal-generation **P100** (sm_60), one of the two GPU types Kaggle itself still offers
here (T4x2 or P100, per the note below). Landing on a P100 crashed training ~40 seconds in with
`CUDA error: no kernel image is available for execution on the device` -- a real run, not a
hypothetical. The cell below detects the actual GPU via `nvidia-smi` (no torch import needed yet,
so this runs before torch's own compute-capability list is fixed for the process) and reinstalls
a CUDA 11.8 build if the assigned GPU isn't in the preinstalled build's supported list -- CUDA 11.8
wheels cover Pascal through Hopper, so this works regardless of which GPU Kaggle happens to assign.

In [ ]:
import subprocess, sys

try:
    cc_raw = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"], text=True
    ).strip().splitlines()[0]
    major, minor = cc_raw.split(".")
    needed_sm = f"sm_{major}{minor}"
except Exception as e:
    needed_sm = None
    print(f"Could not query GPU compute capability via nvidia-smi ({e}) -- skipping the compatibility check.")

if needed_sm:
    # Check the INSTALLED build's supported architectures in a SEPARATE PROCESS, not an in-process
    # `import torch` -- Python caches imports in sys.modules, so even an aliased/deleted in-process
    # import here would make a LATER `import torch` in the next cell silently return the stale
    # cached module instead of a fresh one. Confirmed live: an earlier version of this cell did
    # `import torch as _torch_probe`, and the kernel died ~80s after reinstalling -- the old torch
    # stayed resident in this process while its .so files got replaced out from under it on disk.
    check = subprocess.run(
        [sys.executable, "-c",
         "import torch; print(' '.join(torch.cuda.get_arch_list()) if torch.cuda.is_available() else '')"],
        capture_output=True, text=True,
    )
    supported = check.stdout.split()
    if needed_sm not in supported:
        print(f"GPU needs {needed_sm}, not in the preinstalled torch build's supported list "
              f"{supported} -- reinstalling a CUDA 11.8 build (covers Pascal through Hopper)...")
        # Uninstall the whole torch/torchvision/torchaudio trio first, then install all three
        # together from the SAME cu118 index in one resolution -- reinstalling `torch` alone
        # left mismatched torchvision/nccl versions behind, which crashed two live runs with
        # unrelated-looking errors (undefined symbol ncclCommShrink; aten.OpaqueObject not
        # registered) that were actually both this same root cause from a different angle.
        subprocess.run(["pip", "uninstall", "-y", "-q", "torch", "torchvision", "torchaudio"], check=True)
        subprocess.run(
            ["pip", "install", "-q", "torch", "torchvision", "torchaudio",
             "--index-url", "https://download.pytorch.org/whl/cu118"],
            check=True,
        )
        print("Reinstalled. The check above ran in a subprocess, so THIS process has never "
              "imported torch itself -- the next cell's `import torch` will be a genuinely fresh "
              "import, not a cached one.")
    else:
        print(f"GPU compute capability {needed_sm} already supported by the preinstalled build.")

In [ ]:
import torch
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    print(" -", torch.cuda.get_device_name(i))

## 1. Setup

In [ ]:
!pip install -q segmentation-models-pytorch

In [ ]:
import os, sys, json, math, time, random, glob, collections, shutil
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
from scipy import ndimage

# Local smoke-test hooks -- never set on Kaggle. LC_DATA_ROOT points at a flat folder holding images/*.tif and
# label/*.tif (a few real OpenEarthMap pairs) so the whole notebook can run end to end on a laptop first.
SMOKE_TEST = bool(os.environ.get("LC_SMOKE_TEST"))
DATA_ROOT = os.environ.get("LC_DATA_ROOT")

SEED = 0
CLASSES = ["bareland", "rangeland", "developed space", "road", "tree", "water", "agriculture land", "building"]  # label values 1..8
NUM_CLASSES, IGNORE = len(CLASSES), 255
BUILDING = CLASSES.index("building")
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)  # ImageNet, what the pretrained encoder expects
CROP = 128 if SMOKE_TEST else 512
CROPS_PER_IMAGE = 1 if SMOKE_TEST else 2  # random crops drawn from each training image per epoch
EPOCHS = 1 if SMOKE_TEST else 24
TARGET_BATCH = 2 if SMOKE_TEST else 16
WORKERS = 0 if SMOKE_TEST else 4
LR = 3e-4
MIN_BUILDING_PIXELS = 30  # a connected building region smaller than this (at 0.25-0.5 m/px, ~5-7 m^2) is noise, not a building

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| smoke test:", SMOKE_TEST)

## 2. Find the data and pair every image with its label

The dataset root is found by looking for an `images/` folder with a `train/` subfolder next to a `label/` folder, so a different mount path does not break the notebook. Pairs are matched by file name and every label is checked to contain only the values 0-8 (a wrong encoding would otherwise train silently).

In [ ]:
def find_root():
    if DATA_ROOT:
        return DATA_ROOT
    for images_dir in glob.glob("/kaggle/input/**/images", recursive=True):
        root = os.path.dirname(images_dir)
        if os.path.isdir(os.path.join(root, "label")) and os.path.isdir(os.path.join(images_dir, "train")):
            return root
    raise AssertionError("OpenEarthMap not found under /kaggle/input -- add 'aletbm/global-land-cover-mapping-openearthmap' as an input")

ROOT = find_root()

def list_pairs(split):
    pattern = f"{ROOT}/images/*.tif" if DATA_ROOT else f"{ROOT}/images/{split}/*.tif"
    out = []
    for image in sorted(glob.glob(pattern)):
        label = image.replace("/images/", "/label/")
        if os.path.exists(label):
            out.append((image, label))
    return out

if DATA_ROOT:  # smoke: one flat folder, split by position
    every = list_pairs(None)
    train_pairs, val_pairs = every[:-2], every[-2:]
else:
    train_pairs, val_pairs = list_pairs("train"), list_pairs("val")
assert train_pairs and val_pairs, f"no image/label pairs under {ROOT}"

region = lambda path: os.path.basename(path).rsplit("_", 1)[0]
train_regions, val_regions = {region(p) for p, _ in train_pairs}, {region(p) for p, _ in val_pairs}
print(f"root {ROOT}")
print(f"train {len(train_pairs)} tiles / {len(train_regions)} regions | val {len(val_pairs)} tiles / {len(val_regions)} regions | regions in both: {len(train_regions & val_regions)}")

# The label encoding must be exactly 0 (unlabelled) and 1..8 -- audited on a spread of tiles, with the class shares.
audit = random.Random(SEED).sample(train_pairs, min(60, len(train_pairs)))
counts = np.zeros(256, dtype=np.int64)
for _, label in audit:
    counts += np.bincount(np.array(Image.open(label)).ravel(), minlength=256)
present = np.nonzero(counts)[0].tolist()
assert set(present) <= set(range(0, NUM_CLASSES + 1)), f"unexpected label values {present}"
share = counts / counts.sum()
print("class share of the audited tiles (%):", {("unlabelled" if v == 0 else CLASSES[v - 1]): round(100 * float(share[v]), 1) for v in present})

## 3. Dataset and model

Training crops are taken at a random scale -- a 256-1000 px window (log-uniform) resized to 512 -- so the model sees the scene at ground resolutions from about half to twice the native one; flips and 90-degree rotations; a mild brightness/contrast/saturation jitter only (roof colour is one of the things the app reports, so colours must not be scrambled). Labels are shifted so building = 7 and unlabelled = 255 (ignored by both losses).

In [ ]:
def load_pair(image_path, label_path):
    image = np.array(Image.open(image_path).convert("RGB"))
    label = np.array(Image.open(label_path)).astype(np.int64)
    label = np.where(label == 0, IGNORE, label - 1)  # 1..8 -> 0..7, unlabelled -> 255
    return image, label

def to_tensor(image):
    x = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
    return (x - torch.tensor(MEAN)[:, None, None]) / torch.tensor(STD)[:, None, None]

class TrainCrops(Dataset):
    def __init__(self, pairs, crops):
        self.pairs, self.crops = pairs, crops
    def __len__(self):
        return len(self.pairs) * self.crops
    def __getitem__(self, i):
        image, label = load_pair(*self.pairs[i % len(self.pairs)])
        h, w = label.shape
        side = int(math.exp(random.uniform(math.log(min(256, min(h, w))), math.log(min(h, w)))))
        y, x = random.randint(0, h - side), random.randint(0, w - side)
        image, label = image[y:y + side, x:x + side], label[y:y + side, x:x + side]
        img = torch.from_numpy(image).permute(2, 0, 1).float().unsqueeze(0)
        lab = torch.from_numpy(label).float()[None, None]
        img = F.interpolate(img, size=(CROP, CROP), mode="bilinear", align_corners=False, antialias=side > CROP)[0]
        lab = F.interpolate(lab, size=(CROP, CROP), mode="nearest")[0, 0].long()
        if random.random() < 0.5: img, lab = img.flip(-1), lab.flip(-1)
        if random.random() < 0.5: img, lab = img.flip(-2), lab.flip(-2)
        k = random.randint(0, 3)
        if k: img, lab = torch.rot90(img, k, (-2, -1)), torch.rot90(lab, k, (-2, -1))
        img = img / 255.0
        img = (img - img.mean()) * random.uniform(0.9, 1.1) + img.mean() * random.uniform(0.9, 1.1)  # contrast, brightness
        gray = img.mean(0, keepdim=True)
        img = (gray + (img - gray) * random.uniform(0.9, 1.1)).clamp(0, 1)  # saturation
        img = (img - torch.tensor(MEAN)[:, None, None]) / torch.tensor(STD)[:, None, None]
        return img, lab

def make_loader(dataset, batch, shuffle):
    return DataLoader(dataset, batch_size=batch, shuffle=shuffle, drop_last=shuffle, num_workers=WORKERS,
                      pin_memory=(DEVICE == "cuda"), persistent_workers=(WORKERS > 0))

def build_model():
    return smp.Unet(encoder_name="resnet34", encoder_weights="imagenet", classes=NUM_CLASSES, activation=None)

ce_loss = nn.CrossEntropyLoss(ignore_index=IGNORE)
dice_loss = smp.losses.DiceLoss(mode="multiclass", ignore_index=IGNORE)
def loss_fn(logits, target):
    return ce_loss(logits, target) + 0.5 * dice_loss(logits, target)

## 4. Metrics

Everything is computed from one confusion matrix per pass. The building count is the number of connected regions of the building mask (8-connectivity) with at least `MIN_BUILDING_PIXELS` pixels; touching buildings merge into one region, so it is a count of distinct footprints and under-counts dense terraces -- the notebook measures how far off it is rather than assuming.

In [ ]:
def confusion(pred, target):
    valid = target != IGNORE
    idx = target[valid] * NUM_CLASSES + pred[valid]
    return torch.bincount(idx, minlength=NUM_CLASSES ** 2).reshape(NUM_CLASSES, NUM_CLASSES).cpu().numpy()

def iou_from_confusion(cm):
    tp = np.diag(cm).astype(float)
    denom = cm.sum(0) + cm.sum(1) - np.diag(cm)
    return np.where(denom > 0, tp / np.maximum(denom, 1), np.nan)

def count_components(mask, min_pixels=MIN_BUILDING_PIXELS):
    labelled, n = ndimage.label(mask, structure=np.ones((3, 3)))
    if n == 0:
        return 0
    sizes = np.bincount(labelled.ravel())[1:]
    return int((sizes >= min_pixels).sum())

@torch.no_grad()
def predict_full(model, image):
    """Whole-tile prediction (the network is fully convolutional): reflect-pad to a multiple of 32, predict, crop back."""
    x = to_tensor(image)[None].to(DEVICE)
    h, w = x.shape[-2:]
    x = F.pad(x, (0, (-w) % 32, 0, (-h) % 32), mode="reflect")
    with torch.autocast("cuda", dtype=torch.float16, enabled=(DEVICE == "cuda")):
        logits = model(x)
    return logits.float()[..., :h, :w].argmax(1)[0]

def evaluate(model, pairs, scale=1.0, with_counts=False):
    model.eval()
    cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
    frac_err = np.zeros(NUM_CLASSES); n_tiles = 0
    pred_counts, true_counts = [], []
    for image_path, label_path in pairs:
        image, label = load_pair(image_path, label_path)
        if scale != 1.0:  # simulate a coarser ground resolution: shrink image AND label together
            h, w = label.shape
            size = (int(w * scale), int(h * scale))
            image = np.array(Image.fromarray(image).resize(size, Image.LANCZOS))
            label = np.array(Image.fromarray(label.astype(np.uint8)).resize(size, Image.NEAREST)).astype(np.int64)
        pred = predict_full(model, image)
        target = torch.from_numpy(label).to(DEVICE)
        cm += confusion(pred, target)
        valid = target != IGNORE
        for c in range(NUM_CLASSES):
            frac_err[c] += abs(float(((pred == c) & valid).sum() - ((target == c) & valid).sum())) / max(int(valid.sum()), 1)
        n_tiles += 1
        if with_counts:
            pred_counts.append(count_components((pred == BUILDING).cpu().numpy()))
            true_counts.append(count_components((target == BUILDING).cpu().numpy()))
    iou = iou_from_confusion(cm)
    out = {"mIoU": float(np.nanmean(iou)), "iou": {CLASSES[c]: (None if np.isnan(iou[c]) else float(iou[c])) for c in range(NUM_CLASSES)},
           "pixel_accuracy": float(np.diag(cm).sum() / max(cm.sum(), 1)),
           "area_fraction_error_pct": {CLASSES[c]: 100 * float(frac_err[c] / max(n_tiles, 1)) for c in range(NUM_CLASSES)}}
    if with_counts and true_counts:
        p, t = np.array(pred_counts, float), np.array(true_counts, float)
        out["building_count"] = {"mean_true": float(t.mean()), "mean_pred": float(p.mean()), "mean_abs_error": float(np.abs(p - t).mean()),
                                 "median_relative_error": float(np.median(np.abs(p - t) / np.maximum(t, 1))),
                                 "correlation": float(np.corrcoef(p, t)[0, 1]) if len(t) > 2 and t.std() > 0 and p.std() > 0 else None}
    return out

## 5. Pre-flight: the whole train / save / reload path, before the long run

Round-one of the captioner spent 20 minutes downloading and then died on a library clash, and its second run ran out of memory on the first real step. So this cell runs the *same* functions -- model build with the ImageNet encoder, two fp16-autocast optimiser steps with gradient scaling, whole-tile prediction, save and reload -- on synthetic data first, then **finds the largest batch that fits** (one worst-case forward+backward per candidate, 15% of GPU memory kept free).

In [ ]:
def preflight():
    t0 = time.time()
    use_amp = DEVICE == "cuda"
    m = build_model().to(DEVICE)
    opt = torch.optim.AdamW(m.parameters(), lr=1e-4)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    x = torch.randn(2, 3, CROP, CROP, device=DEVICE)
    y = torch.randint(0, NUM_CLASSES, (2, CROP, CROP), device=DEVICE); y[:, :4] = IGNORE
    m.train()
    for _ in range(2):
        with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
            loss = loss_fn(m(x), y)
        assert torch.isfinite(loss), f"non-finite loss ({float(loss)}) -- fp16 is overflowing"
        scaler.scale(loss).backward(); scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
        scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
    pred = predict_full(m, np.random.randint(0, 255, (300, 260, 3), dtype=np.uint8))
    assert tuple(pred.shape) == (300, 260) and int(pred.max()) < NUM_CLASSES
    path = "/tmp/preflight_lc.pt"
    torch.save({"model_state_dict": m.state_dict()}, path)
    m2 = build_model(); m2.load_state_dict(torch.load(path, map_location="cpu")["model_state_dict"]); os.remove(path)
    del m, m2, opt
    if DEVICE == "cuda": torch.cuda.empty_cache()
    print(f"pre-flight OK in {time.time() - t0:.0f}s: model build, fp16 train steps, tile prediction, save/reload all run in THIS environment")

preflight()

def pick_batch(target):
    if DEVICE != "cuda":
        return target
    m = build_model().to(DEVICE); m.train()
    scaler = torch.amp.GradScaler("cuda")
    total = torch.cuda.get_device_properties(0).total_memory
    chosen = None
    for cand in [c for c in (24, 16, 12, 8, 4, 2, 1) if c <= max(target, 8)]:
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
        try:
            x = torch.randn(cand, 3, CROP, CROP, device=DEVICE)
            y = torch.randint(0, NUM_CLASSES, (cand, CROP, CROP), device=DEVICE)
            with torch.autocast("cuda", dtype=torch.float16):
                loss = loss_fn(m(x), y)
            scaler.scale(loss).backward()
            peak = torch.cuda.max_memory_allocated()
        except torch.cuda.OutOfMemoryError:
            peak = total
        for p in m.parameters(): p.grad = None
        x = y = loss = None
        print(f"  batch {cand:2d}: peak {peak / 1e9:5.1f} GB of {total / 1e9:.1f} GB -> {'fits' if peak < 0.85 * total else 'too big'}")
        if peak < 0.85 * total:
            chosen = cand; break
    assert chosen, "not even batch 1 fits"
    del m, scaler
    torch.cuda.empty_cache()
    return chosen

BATCH = pick_batch(TARGET_BATCH)
print("batch size:", BATCH)

## 6. Train

AdamW, warm-up then cosine decay, fp16 autocast + gradient scaling. Every epoch the model is scored on a fixed subset of 128 validation tiles (the whole 384 would cost a further ~40 s per epoch) and the weights with the best mIoU are kept; the complete validation set is scored once at the end. A non-finite loss skips the step, and more than 5% of them aborts the run.

In [ ]:
train_loader = make_loader(TrainCrops(train_pairs, CROPS_PER_IMAGE), BATCH, shuffle=True)
val_subset = random.Random(SEED).sample(val_pairs, min(128, len(val_pairs)))
model = build_model().to(DEVICE)
encoder_params = list(model.encoder.parameters())
encoder_ids = {id(p) for p in encoder_params}
other_params = [p for p in model.parameters() if id(p) not in encoder_ids]
optimizer = torch.optim.AdamW([{"params": encoder_params, "lr": LR / 3}, {"params": other_params, "lr": LR}], weight_decay=1e-4)
steps_per_epoch = len(train_loader)
steps_total = EPOCHS * steps_per_epoch
warmup = max(1, int(0.03 * steps_total))
def lr_factor(step):
    if step < warmup:
        return (step + 1) / warmup
    progress = (step - warmup) / max(1, steps_total - warmup)
    return 0.02 + 0.98 * 0.5 * (1 + math.cos(math.pi * progress))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
use_amp = DEVICE == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

CKPT_DIR = "/tmp/lc_ckpt" if SMOKE_TEST else "/kaggle/working/lc_ckpt"
os.makedirs(CKPT_DIR, exist_ok=True)
best_miou, best_state, skipped, step, history = -1.0, None, 0, 0, []
t_start = time.time()
print(f"{steps_total} optimiser steps ({steps_per_epoch} per epoch), batch {BATCH}")
for epoch in range(EPOCHS):
    model.train()
    running, n, t_epoch = 0.0, 0, time.time()
    for images, labels in train_loader:
        images, labels = images.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
            loss = loss_fn(model(images), labels)
        if not torch.isfinite(loss):
            skipped += 1
            optimizer.zero_grad(set_to_none=True)
            assert skipped <= max(3, 0.05 * (step + 1)), "too many non-finite losses -- fp16 is overflowing"
            continue
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update(); scheduler.step()
        running += loss.item(); n += 1; step += 1
        if step % 100 == 0:
            print(f"  step {step}/{steps_total}  loss {running / max(n, 1):.4f}  ({time.time() - t_start:.0f}s)", flush=True)
    result = evaluate(model, val_subset)
    history.append({"epoch": epoch + 1, "train_loss": running / max(n, 1), "val_mIoU": result["mIoU"]})
    marker = ""
    if result["mIoU"] > best_miou:
        best_miou, best_state, marker = result["mIoU"], {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}, "  <- best"
        torch.save({"model_state_dict": best_state}, os.path.join(CKPT_DIR, "best_state.pt"))
    print(f"epoch {epoch + 1}/{EPOCHS}  train loss {running / max(n, 1):.4f}  val mIoU {result['mIoU']:.4f}  ({time.time() - t_epoch:.0f}s){marker}", flush=True)
print(f"\ntraining took {(time.time() - t_start) / 60:.1f} min; best subset mIoU {best_miou:.4f}; skipped {skipped} non-finite steps")
model.load_state_dict(best_state)

## 7. Final evaluation

The complete validation set with the best weights, at native scale (with the building-count comparison) and at half scale (a coarser ground resolution, closer to a wide-area capture).

In [ ]:
final = evaluate(model, val_pairs, scale=1.0, with_counts=True)
half = evaluate(model, val_pairs, scale=0.5)
print(f"\nVALIDATION ({len(val_pairs)} tiles)  mIoU {final['mIoU']:.4f}  pixel accuracy {final['pixel_accuracy']:.4f}   |   half scale: mIoU {half['mIoU']:.4f}")
print(f"  {'class':18s} {'IoU':>7s} {'IoU @0.5x':>10s} {'area-fraction error (pct points of the scene)':>48s}")
for c in CLASSES:
    a, b = final["iou"][c], half["iou"][c]
    print(f"  {c:18s} {('n/a' if a is None else f'{a:.3f}'):>7s} {('n/a' if b is None else f'{b:.3f}'):>10s} {final['area_fraction_error_pct'][c]:>48.2f}")
bc = final.get("building_count")
if bc:
    print(f"\nbuilding count per tile: true {bc['mean_true']:.1f}, predicted {bc['mean_pred']:.1f}, mean abs error {bc['mean_abs_error']:.1f}, "
          f"median relative error {bc['median_relative_error']:.0%}, correlation {bc['correlation'] if bc['correlation'] is None else round(bc['correlation'], 3)}")

## 8. Export

One checkpoint dict with everything `models/landcover/landcover_tool.py` needs to rebuild and run the model (encoder name, class names, normalisation, the training crop size) plus the metrics, and a JSON copy of the metrics.

In [ ]:
OUT_DIR = "/tmp/lc_out" if SMOKE_TEST else "/kaggle/working/lc_out"
os.makedirs(OUT_DIR, exist_ok=True)
metrics = {"val_tiles": len(val_pairs), "native": final, "half_scale": half, "history": history, "epochs": EPOCHS, "train_tiles": len(train_pairs)}
torch.save({
    "model_state_dict": best_state, "encoder_name": "resnet34", "classes": CLASSES, "num_classes": NUM_CLASSES,
    "train_crop": CROP, "mean": MEAN, "std": STD, "metrics": metrics,
    "label_note": "OpenEarthMap labels 1..8 map to classes 0..7 in this order; 0 = unlabelled (ignored)",
}, os.path.join(OUT_DIR, "landcover_unet.pt"))
json.dump(metrics, open(os.path.join(OUT_DIR, "landcover_metrics.json"), "w"), indent=1)
print("exported:", sorted(os.listdir(OUT_DIR)), f"({os.path.getsize(os.path.join(OUT_DIR, 'landcover_unet.pt')) / 1e6:.0f} MB)")

# reload the SAVED artifact and predict once, so a broken export fails here and not on the laptop
ck = torch.load(os.path.join(OUT_DIR, "landcover_unet.pt"), map_location="cpu", weights_only=False)
check = build_model(); check.load_state_dict(ck["model_state_dict"]); check.to(DEVICE)
image, label = load_pair(*val_pairs[0])
pred = predict_full(check, image)
print("artifact round-trip OK:", tuple(pred.shape), "predicted classes:", sorted(set(pred.cpu().numpy().ravel().tolist())))